# Common Libraries

In [1]:
import os, mujoco
import numpy as np
import ipywidgets as widgets
from mink import Configuration, SE3, SO3, FrameTask
from IPython.display import display, clear_output
from scipy.spatial.transform import Rotation as R

# Model specification

In [2]:
dir_path = "/Users/seojin/mink/examples"
model_path = os.path.join(dir_path, "franka_emika_panda/mjx_scene.xml")
model = mujoco.MjModel.from_xml_path(model_path)

# Simulation

In [3]:
configuration = Configuration(model)
configuration.update_from_keyframe("home")

renderer = mujoco.Renderer(configuration.model, height=400, width=600)

task = FrameTask(
    frame_name="attachment_site",
    frame_type="site",
    position_cost=1.0,
    orientation_cost=1.0,
)

In [4]:
def update_robot(x, y, z, roll, pitch, yaw):
    # Target
    new_translation = np.array([x, y, z])
    r = R.from_euler('xyz', [roll, pitch, yaw], degrees=True)
    new_quat = r.as_quat() # 결과는 [x, y, z, w] 순서
    mink_quat = np.array([new_quat[3], new_quat[0], new_quat[1], new_quat[2]])
    target_val = np.concatenate([mink_quat, new_translation])
    new_target = SE3(wxyz_xyz = target_val)

    # Render target information
    formatted_text = ", ".join([f"{val:.4f}" for val in target_val])
    data_label.value = f"wxyz_xyz: [{formatted_text}]"

    # Set target position and orientation
    task.set_target(new_target)
    configuration.data.mocap_pos[0] = new_target.wxyz_xyz[4:]
    configuration.data.mocap_quat[0] = new_target.wxyz_xyz[:4]
    
    # Rendering
    mujoco.mj_forward(configuration.model, configuration.data)
    renderer.update_scene(configuration.data)
    pixels = renderer.render()
    
    with output_area:
        clear_output(wait=True)
        import mediapy as media
        media.show_image(pixels)

# Make widgets
data_label = widgets.Label(value="wxyz_xyz: [0, 0, 0, 0, 0, 0, 0]")
x_slider = widgets.FloatSlider(value=0.5, min=0.2, max=0.8, step=0.01, description='Target X:')
y_slider = widgets.FloatSlider(value=-0.4, min=-0.8, max=0.8, step=0.01, description='Target Y:')
z_slider = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.01, description='Target Z:')

roll_slider = widgets.FloatSlider(value=0, min=-180, max=180, step=1, description='Roll (X)')
pitch_slider = widgets.FloatSlider(value=0, min=-180, max=180, step=1, description='Pitch (Y)')
yaw_slider = widgets.FloatSlider(value=0, min=-180, max=180, step=1, description='Yaw (Z)')

output_area = widgets.Output()

# Layout and event connection
ui = widgets.VBox([
    data_label,
    widgets.HBox([x_slider, roll_slider]),
    widgets.HBox([y_slider, pitch_slider]),
    widgets.HBox([z_slider, yaw_slider])
])

out = widgets.interactive_output(update_robot, {
    'x': x_slider, 'y': y_slider, 'z': z_slider,
    'roll': roll_slider, 'pitch': pitch_slider, 'yaw': yaw_slider
})

display(ui, output_area)
update_robot(x_slider.value, y_slider.value, z_slider.value, roll_slider.value, pitch_slider.value, yaw_slider.value)

Output()